# 03 — Synthetic Experiment Analysis

Deep analysis of demo-selection performance across controlled distribution shifts.

**Key questions:**
1. Does demo selection improve OOD accuracy compared to zero-shot and random?
2. Which selection strategies are most robust under each shift type?
3. How does the ID→OOD accuracy gap vary by condition and shift type?

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))
from src.evaluation.accuracy import accuracy, macro_f1, invalid_rate

sns.set_theme(style="whitegrid", font_scale=1.1)

RESULTS_DIR = Path.cwd().parent / "results" / "v2" / "synthetic_baselines"
results = pd.read_parquet(RESULTS_DIR / "synthetic_baselines_qwen7b.parquet")
print(f"Loaded {len(results)} rows")
results.head()

## 1. Accuracy heatmap: dataset × condition × environment

In [ ]:
def compute_metrics(df):
    return pd.Series({
        "accuracy": accuracy(df["prediction"], df["label"]),
        "macro_f1": macro_f1(df["prediction"], df["label"]),
        "invalid_rate": invalid_rate(df["prediction"]),
    })

metrics = results.groupby(["dataset", "method", "environment"]).apply(
    compute_metrics, include_groups=False
).reset_index()

# Accuracy pivot
for env in ["id", "ood"]:
    print(f"\n{'=' * 60}")
    print(f"Accuracy — {env.upper()}")
    print("=" * 60)
    sub = metrics[metrics["environment"] == env]
    piv = sub.pivot(index="dataset", columns="method", values="accuracy")
    print(piv.to_string(float_format="%.3f"))

In [ ]:
# Heatmap visualisation
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for i, env in enumerate(["id", "ood"]):
    sub = metrics[metrics["environment"] == env]
    piv = sub.pivot(index="dataset", columns="method", values="accuracy")
    sns.heatmap(
        piv, annot=True, fmt=".3f", cmap="RdYlGn",
        vmin=0.3, vmax=0.9, ax=axes[i],
        cbar_kws={"label": "Accuracy"},
    )
    axes[i].set_title(f"{env.upper()} Accuracy", fontsize=14)
    axes[i].set_ylabel("")

plt.tight_layout()
plt.savefig(str(Path.cwd().parent / "figures" / "v2" / "synth_accuracy_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()

## 2. Shift gap analysis (ID acc − OOD acc)

In [ ]:
id_acc = metrics[metrics["environment"] == "id"].set_index(["dataset", "method"])["accuracy"]
ood_acc = metrics[metrics["environment"] == "ood"].set_index(["dataset", "method"])["accuracy"]
gap = (id_acc - ood_acc).reset_index(name="shift_gap")

gap_pivot = gap.pivot(index="dataset", columns="method", values="shift_gap")
print("Shift gap (ID − OOD accuracy):")
print("Positive = ID better, Negative = OOD better")
print(gap_pivot.to_string(float_format="%.3f"))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
gap_melted = gap.copy()
sns.barplot(
    data=gap_melted, x="method", y="shift_gap", hue="dataset",
    ax=ax, palette="Set2",
)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Demo Selection Condition")
ax.set_ylabel("Shift Gap (ID acc − OOD acc)")
ax.set_title("Shift gap by condition and shift type")
ax.tick_params(axis="x", rotation=30)
ax.legend(title="Dataset")
plt.tight_layout()
plt.savefig(str(Path.cwd().parent / "figures" / "v2" / "synth_shift_gap.png"), dpi=150, bbox_inches="tight")
plt.show()

## 3. Condition improvement over zero-shot

In [ ]:
# Compute improvement over zero-shot baseline for OOD
ood_metrics = metrics[metrics["environment"] == "ood"].copy()

zs_acc = ood_metrics[ood_metrics["method"] == "zero_shot"].set_index("dataset")["accuracy"]

ood_metrics["improvement"] = ood_metrics.apply(
    lambda r: r["accuracy"] - zs_acc.get(r["dataset"], 0), axis=1
)

non_zs = ood_metrics[ood_metrics["method"] != "zero_shot"]

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(
    data=non_zs, x="method", y="improvement", hue="dataset",
    ax=ax, palette="Set2",
)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Demo Selection Condition")
ax.set_ylabel("OOD Accuracy Improvement over Zero-Shot")
ax.set_title("Does demo selection help under distribution shift?")
ax.tick_params(axis="x", rotation=30)
ax.legend(title="Shift Type")
plt.tight_layout()
plt.savefig(str(Path.cwd().parent / "figures" / "v2" / "synth_improvement.png"), dpi=150, bbox_inches="tight")
plt.show()

print("\nMean OOD improvement over zero-shot:")
print(non_zs.pivot(index="dataset", columns="method", values="improvement").to_string(float_format="%.3f"))

## 4. Confidence distribution analysis

In [ ]:
if "confidence" in results.columns:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for i, ds_name in enumerate(results["dataset"].unique()):
        sub = results[(results["dataset"] == ds_name) & (results["method"] != "zero_shot")]
        for env in ["id", "ood"]:
            env_sub = sub[sub["environment"] == env]
            axes[i].hist(
                env_sub["confidence"].dropna(), bins=20, alpha=0.5,
                density=True, label=env.upper(),
            )
        axes[i].set_title(ds_name)
        axes[i].set_xlabel("Confidence")
        axes[i].legend()

    fig.suptitle("Model confidence: ID vs OOD (few-shot conditions)", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("No confidence column in results.")

## 5. Per-dataset summary

In [ ]:
for ds_name in results["dataset"].unique():
    ds = results[results["dataset"] == ds_name]
    print(f"\n{'=' * 60}")
    print(f"{ds_name}")
    print("=" * 60)

    summary = ds.groupby(["method", "environment"]).apply(
        compute_metrics, include_groups=False
    ).reset_index()

    for env in ["id", "ood"]:
        env_sum = summary[summary["environment"] == env].set_index("method")
        print(f"\n  {env.upper()}:")
        print(env_sum[["accuracy", "macro_f1", "invalid_rate"]].to_string(float_format="%.3f"))

    # Best condition for OOD
    ood_sum = summary[summary["environment"] == "ood"].set_index("method")
    best = ood_sum["accuracy"].idxmax()
    print(f"\n  Best OOD condition: {best} (acc={ood_sum.loc[best, 'accuracy']:.3f})")

## 6. Key findings

In [ ]:
print("KEY FINDINGS")
print("=" * 60)

for ds_name in results["dataset"].unique():
    ds_ood = results[(results["dataset"] == ds_name) & (results["environment"] == "ood")]
    ds_id = results[(results["dataset"] == ds_name) & (results["environment"] == "id")]

    zs_ood = accuracy(
        ds_ood[ds_ood["method"] == "zero_shot"]["prediction"],
        ds_ood[ds_ood["method"] == "zero_shot"]["label"],
    )
    rand_ood = accuracy(
        ds_ood[ds_ood["method"] == "random"]["prediction"],
        ds_ood[ds_ood["method"] == "random"]["label"],
    )

    best_method = None
    best_acc = 0
    for method in ds_ood["method"].unique():
        if method in ["zero_shot", "random"]:
            continue
        acc = accuracy(
            ds_ood[ds_ood["method"] == method]["prediction"],
            ds_ood[ds_ood["method"] == method]["label"],
        )
        if acc > best_acc:
            best_acc = acc
            best_method = method

    print(f"\n{ds_name}:")
    print(f"  Zero-shot OOD:        {zs_ood:.3f}")
    print(f"  Random OOD:           {rand_ood:.3f}")
    print(f"  Best selection OOD:   {best_acc:.3f} ({best_method})")
    print(f"  Selection vs random:  {best_acc - rand_ood:+.3f}")
    print(f"  Selection vs zero:    {best_acc - zs_ood:+.3f}")